# Generate heatmaps for alignment matrices

In [31]:
"""
importing libraries
"""
from datetime import datetime
import pandas as pd
from pathlib import Path
import pickle as pl
import numpy as np
import seaborn as sns  
import matplotlib.pyplot as plt
import json
from datasets import load_dataset
import re

In [ ]:
# =========================================================
# DATASET LOADING
# =========================================================
def load_train_dataset(dataset_name: str):
    """
    Load the training split of a supported HuggingFace dataset.

    Args:
        dataset_name (str): Dataset identifier (ag_news, snli, yelp_review, yelp_review_full).

    Returns:
        Dataset: Training split of the requested dataset.

    Raises:
        ValueError: If dataset_name is not supported.
    """
    if dataset_name == "ag_news":
        return load_dataset("ag_news", split="train")
    if dataset_name == "snli":
        return load_dataset("snli", split="train")
    if dataset_name in {"yelp_review", "yelp_review_full"}:
        return load_dataset("yelp_review_full", split="train")
    raise ValueError(f"Unsupported dataset_name: {dataset_name}")


def get_class_names(dataset_name: str, num_classes: int):
    """
    Return human-readable class labels for a given dataset.

    Args:
        dataset_name (str): Dataset identifier.
        num_classes (int): Number of classes (used as fallback).

    Returns:
        List[str]: List of class names.
    """
    class_names_map = {
        "ag_news": ["World", "Sports", "Business", "Sci/Tech"],
        "snli": ["Entailment", "Neutral", "Contradiction"],
        "yelp_review": ["1★", "2★", "3★", "4★", "5★"],
        "yelp_review_full": ["1★", "2★", "3★", "4★", "5★"],
    }
    return class_names_map.get(dataset_name, [f"C{i}" for i in range(num_classes)])


# =========================================================
# PARSING
# =========================================================
def extract_proportion_tuple(name: str):
    """
    Extract a tuple of floats from a string containing values in
    square brackets [] or parentheses ().

    Args:
        name (str): Input string (e.g., "exp_[0.2, 0.3, 0.5]").

    Returns:
        tuple[float] or None: Extracted values as a tuple, or None if no match.
    """
    m = re.search(r"(\[[^\]]+\]|\([^)]*\))", name)
    if not m:
        return None
    raw = m.group(1)[1:-1]
    return tuple(float(x.strip()) for x in raw.split(",") if x.strip())


def parse_align_dir_name(name: str, dataset_name: str):
    """
    Parse directory name to extract dataset, method, and proportion tuple.

    Args:
        name (str): Directory name containing dataset, method, and proportions.
        dataset_name (str): Expected dataset prefix.

    Returns:
        dict or None: Parsed components {"dataset", "method", "proportion"},
        or None if format is invalid.
    """
    prop = extract_proportion_tuple(name)
    if prop is None:
        return None

    # Remove the proportion part
    prefix = re.sub(r"(\[[^\]]+\]|\([^)]*\))", "", name).rstrip("_")

    expected_prefix = f"{dataset_name}_"
    if not prefix.startswith(expected_prefix):
        return None

    method = prefix[len(expected_prefix):]
    if not method:
        return None

    return {
        "dataset": dataset_name,
        "method": method,
        "proportion": prop,
    }


def parse_datainfo_dir_name(name: str):
    """
    Parse directory name to extract dataset and proportion tuple.

    Args:
        name (str): Directory name (e.g., "ag_news_(...)").

    Returns:
        dict or None: {"dataset", "proportion"} or None if parsing fails.
    """
    prop = extract_proportion_tuple(name)
    if prop is None:
        return None

    prefix = name.split("(")[0].rstrip("_")
    return {
        "dataset": prefix,
        "proportion": prop,
    }


# =========================================================
# MATCHING
# =========================================================
def collect_alignment_dirs(align_root: Path, dataset_name: str, method: str | None = None):
    """
    Collect alignment directories mapped by proportion tuples.

    Args:
        align_root (Path): Root directory containing alignment subdirs.
        dataset_name (str): Dataset prefix to filter directories.
        method (str | None): Optional method filter.

    Returns:
        dict: {proportion_tuple: Path} for matching directories.
    """
    out = {}
    for p in align_root.iterdir():
        if not p.is_dir():
            continue
        parsed = parse_align_dir_name(p.name, dataset_name)
        if parsed is None:
            continue
        if method is not None and parsed["method"] != method:
            continue
        out[parsed["proportion"]] = p
    return out


def collect_datainfo_dirs(datainfo_root: Path, dataset_name: str):
    """
    Collect datainfo directories mapped by proportion tuples.

    Args:
        datainfo_root (Path): Root directory containing datainfo subdirs.
        dataset_name (str): Dataset prefix to filter directories.

    Returns:
        dict: {proportion_tuple: Path} for matching directories.
    """
    out = {}
    for p in datainfo_root.iterdir():
        if not p.is_dir():
            continue
        parsed = parse_datainfo_dir_name(p.name)
        if parsed is None:
            continue
        if parsed["dataset"] != dataset_name:
            continue
        out[parsed["proportion"]] = p
    return out


def get_matched_pairs( align_root,datainfo_root,dataset_name,method,selected_proportions=None):
    """
    Match alignment and datainfo directories by common proportion tuples.

    Args:
        align_root (Path | str): Root of alignment directories.
        datainfo_root (Path | str): Root of datainfo directories.
        dataset_name (str): Dataset to filter.
        method (str): Alignment method to filter.
        selected_proportions (list | None): Optional subset of proportions.
        max_pairs (int | None): Optional limit on number of pairs.

    Returns:
        list[tuple]: [(proportion, align_path, datainfo_path), ...]
    """
    align_root = Path(align_root)
    datainfo_root = Path(datainfo_root)

    align_dirs = collect_alignment_dirs(align_root, dataset_name, method=method)
    datainfo_dirs = collect_datainfo_dirs(datainfo_root, dataset_name)

    common_props = sorted(set(align_dirs) & set(datainfo_dirs))

    if selected_proportions is not None:
        selected_set = {tuple(map(float, p)) for p in selected_proportions}
        common_props = [p for p in common_props if p in selected_set]

    return [(prop, align_dirs[prop], datainfo_dirs[prop]) for prop in common_props]


# =========================================================
# CORE COMPUTATION
# =========================================================
def compute_per_class_alignment(alignment_matrix: np.ndarray, labels: np.ndarray) -> np.ndarray:
    """
    Compute mean alignment scores per class for each pseudo-expert.

    Args:
        alignment_matrix (np.ndarray): Shape (N, K) with alignment scores.
        labels (np.ndarray): Shape (N,) with class labels.

    Returns:
        np.ndarray: Shape (K, C) where each entry is the mean alignment
        of expert k for class c (NaN if no samples for class).
    """
    if alignment_matrix.ndim != 2:
        raise ValueError(f"Expected 2D alignment_matrix, got shape {alignment_matrix.shape}")

    N, K = alignment_matrix.shape
    if N != len(labels):
        raise ValueError(
            f"Mismatch: alignment_matrix has {N} samples, but labels has {len(labels)} entries."
        )

    num_classes = int(labels.max()) + 1
    out = np.full((K, num_classes), np.nan, dtype=np.float32)

    for k in range(K):
        for c in range(num_classes):
            mask = labels == c
            if np.any(mask):
                out[k, c] = alignment_matrix[mask, k].mean()

    return out


def get_labels_from_datainfo(dataset, datainfo_json_path: Path):
    """
    Extract labels for selected indices from a dataset using a datainfo JSON.

    Args:
        dataset: Dataset object with "label" field.
        datainfo_json_path (Path): Path to JSON containing "indices_D".

    Returns:
        tuple: (labels array, dataset_info dict)
    """
    with open(datainfo_json_path, "r") as f:
        dataset_info = json.load(f)

    valid_indices = dataset_info["indices_D"]
    labels = np.array([dataset[idx]["label"] for idx in valid_indices], dtype=np.int64)
    return labels, dataset_info


# =========================================================
# FILE DISCOVERY
# =========================================================
def find_alignment_matrix_file(align_dir: Path, method="linear"):
    preferred = align_dir / f"alignment_matrix_{method}.npy"
    if preferred.exists():
        return preferred

    generic = sorted(align_dir.glob("alignment_matrix*.npy"))
    if generic:
        if len(generic) > 1:
            print(f"Warning: multiple alignment files in {align_dir}, using {generic[0].name}")
        return generic[0]

    raise FileNotFoundError(f"No alignment matrix file found in {align_dir}")


# =========================================================
# HEATMAP
# =========================================================
def save_alignment_heatmap( per_class_alignment: np.ndarray, class_proportions: np.ndarray, dataset_name: str, save_path: Path, title: str | None = None,):
    """
    Generate and save a heatmap of per-class alignment scores.

    Args:
        per_class_alignment (np.ndarray): Shape (K, C) alignment matrix.
        class_proportions (np.ndarray): Class distribution proportions.
        dataset_name (str): Dataset identifier for class labels.
        save_path (Path): Output path for the heatmap image.
        title (str | None): Optional plot title.
    """
    num_classes = per_class_alignment.shape[1]
    K = per_class_alignment.shape[0]

    class_names = get_class_names(dataset_name, num_classes)
    yticklabels = [
        f"{class_names[i]} ({100 * class_proportions[i]:.1f}%)"
        for i in range(num_classes)
    ]

    plt.figure(figsize=(12, 8))
    sns.heatmap(
        per_class_alignment.T,
        annot=True,
        fmt=".3f",
        cmap="RdYlGn",
        xticklabels=[f"θ_{i}" for i in range(K)],
        yticklabels=yticklabels,
        cbar_kws={"label": "Mean Alignment Score"},
        linewidths=0.5,
    )

    plt.xlabel("Pseudo-Expert Index", fontsize=12)
    plt.ylabel("Class", fontsize=12)
    plt.title(
        title or "Mean Alignment Score per Class across Pseudo-Experts",
        fontsize=14,
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight", format="png")
    plt.close()


# =========================================================
# GENERATE ONE
# =========================================================
def generate_one_per_class_alignment( align_dir: Path, datainfo_dir: Path, dataset, output_root: Path, dataset_name, method, 
output_filename="per_class_alignment.npy", 
metadata_filename="metadata.json",
heatmap_filename="per_class_alignment_heatmap.png"):
    """
    Generate, save, and visualize per-class alignment for one matched pair
    of alignment and datainfo directories.

    Args:
        align_dir (Path): Directory containing alignment matrix file.
        datainfo_dir (Path): Directory containing dataset info JSON.
        dataset: Dataset object used to fetch labels.
        output_root (Path): Root directory for saving outputs.
        dataset_name (str): Dataset identifier.
        method (str): Alignment method name.
        datainfo_filename (str): Name of dataset info JSON file.
        output_filename (str): Name of saved alignment matrix file.
        metadata_filename (str): Name of saved metadata JSON file.
        heatmap_filename (str): Name of saved heatmap image.

    Returns:
        dict: Paths and metadata for generated outputs.
    """
    datainfo_path = datainfo_dir / "dataset_info.json"

    alignment_path = find_alignment_matrix_file(align_dir, method=method)
    alignment_matrix = np.load(alignment_path)

    labels, dataset_info = get_labels_from_datainfo(dataset, datainfo_path)
    per_class_alignment = compute_per_class_alignment(alignment_matrix, labels)

    num_classes = per_class_alignment.shape[1]
    class_counts = np.bincount(labels, minlength=num_classes)
    class_proportions = class_counts / class_counts.sum()

    out_dir = output_root / datainfo_dir.name
    out_dir.mkdir(parents=True, exist_ok=True)

    np.save(out_dir / output_filename, per_class_alignment)

    save_alignment_heatmap(
        per_class_alignment=per_class_alignment,
        class_proportions=class_proportions,
        dataset_name=dataset_name,
        save_path=out_dir / heatmap_filename,
        title=f"Per-Class Alignment Heatmap\n{datainfo_dir.name}",
    )

    metadata = {
        "dataset_name": dataset_name,
        "method": method,
        "proportion_key": list(extract_proportion_tuple(datainfo_dir.name)),
        "alignment_dir": str(align_dir),
        "alignment_file_used": str(alignment_path),
        "datainfo_dir": str(datainfo_dir),
        "alignment_shape": list(alignment_matrix.shape),
        "per_class_alignment_shape": list(per_class_alignment.shape),
        "class_counts": class_counts.tolist(),
        "class_proportions": class_proportions.tolist(),
        "indices_count": int(len(dataset_info["indices_D"])),
        "saved_files": {
            "matrix": output_filename,
            "heatmap": heatmap_filename,
            "metadata": metadata_filename,
        },
    }

    with open(out_dir / metadata_filename, "w") as f:
        json.dump(metadata, f, indent=2)

    return {
        "proportion_key": extract_proportion_tuple(datainfo_dir.name),
        "align_dir": align_dir,
        "datainfo_dir": datainfo_dir,
        "alignment_file": alignment_path,
        "output_dir": out_dir,
        "heatmap_path": out_dir / heatmap_filename,
    }


# =========================================================
# DRIVER
# =========================================================
def generate_per_class_alignments(align_root, datainfo_root, output_root, dataset_name, method, selected_proportions=None):
    """
    Run per-class alignment generation across all matched proportion directories.

    Loads dataset, matches alignment/datainfo dirs, computes per-class alignment,
    saves results (matrix, heatmap, metadata), and logs progress.

    Args:
        align_root (Path | str): Root of alignment directories.
        datainfo_root (Path | str): Root of datainfo directories.
        output_root (Path | str): Directory to save outputs.
        dataset_name (str): Dataset identifier.
        method (str): Alignment method filter.
        selected_proportions (list | None): Optional subset of proportions.

    Returns:
        list[dict]: Results for each processed proportion.
    """
    align_root = Path(align_root)
    datainfo_root = Path(datainfo_root)
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    dataset = load_train_dataset(dataset_name)

    matched_pairs = get_matched_pairs(
        align_root=align_root,
        datainfo_root=datainfo_root,
        dataset_name=dataset_name,
        method=method,
        selected_proportions=selected_proportions
    )

    if not matched_pairs:
        raise RuntimeError(
            f"No matching proportion directories found for dataset={dataset_name}, method={method}"
        )

    print(f"Found {len(matched_pairs)} matched pairs for method={method}.\n")

    results = []
    for proportion_key, align_dir, datainfo_dir in matched_pairs:
        print(f"Processing proportion: {proportion_key}")
        result = generate_one_per_class_alignment(
            align_dir=align_dir,
            datainfo_dir=datainfo_dir,
            dataset=dataset,
            output_root=output_root,
            dataset_name=dataset_name,
            method=method,
        )
        results.append(result)
        print(f"  alignment file: {result['alignment_file'].name}")
        print(f"  saved matrix  : {result['output_dir'] / 'per_class_alignment.npy'}")
        print(f"  saved heatmap : {result['heatmap_path']}\n")

    return results

In [ ]:
method = "model_baseline"
dataset_name = "ag_news"
align_path = f"./results_align_matrix/results_align_matrix_{dataset_name}"
datainfo_path = f"./results_datainfo/results_datainfo_{dataset_name}"

results = generate_per_class_alignments(
    align_root = align_path ,
    datainfo_root = datainfo_path,
    method = method,
    output_root = f"./results_align_matrix_heatmaps/results_per_class_alignment_{dataset_name}_{method}",
    dataset_name = dataset_name
)